In [3]:
# STEP 6: Download Dataset 1 — SST-2 Sentiment (Primary)

from datasets import load_dataset
import pandas as pd

# Load SST-2
dataset = load_dataset('glue', 'sst2')

# Take a small subset to simulate low-resource setting
train_df = dataset['train'].to_pandas().sample(n=800, random_state=42)
val_df = dataset['validation'].to_pandas()

# Save
train_df.to_csv('../Data/raw/sst2_train.csv', index=False)
val_df.to_csv('../Data/raw/sst2_val.csv', index=False)

print(f'Training samples: {len(train_df)}')
print(f'Label distribution: {train_df.label.value_counts().to_dict()}')

Training samples: 800
Label distribution: {1: 432, 0: 368}


In [ ]:
# STEP 7: Balanced Download (Civil Comments)

from datasets import load_dataset
import pandas as pd
import os

print("Downloading Civil Comments dataset...")
# Using streaming to keep your environment lean
dataset2 = load_dataset("google/civil_comments", split='train', streaming=True)

# 1. Increase stream to 10,000 to find enough toxic samples
data_list = []
for i, example in enumerate(dataset2):
    data_list.append(example)
    if i >= 10000: break

jigsaw_df = pd.DataFrame(data_list)

# 2. Map Column Names
if 'text' in jigsaw_df.columns:
    jigsaw_df = jigsaw_df.rename(columns={'text': 'comment_text'})

# 3. Handle Toxicity (Standard 0.5 threshold)
jigsaw_df['toxic'] = (jigsaw_df['toxicity'] >= 0.5).astype(int)

# 4. Verify Identity Columns
identity_cols = ['male', 'female', 'black', 'white']
available_identities = [col for col in identity_cols if col in jigsaw_df.columns]
keep_cols = ['comment_text', 'toxic'] + available_identities

# 5. Balancing the Dataset (Target: 400 Toxic / 400 Non-Toxic)
toxic = jigsaw_df[jigsaw_df['toxic'] == 1]
non_toxic = jigsaw_df[jigsaw_df['toxic'] == 0]

# Determine the sample size (n=400 or the maximum available toxic samples)
n = min(len(toxic), 400)
print(f"Creating a balanced dataset with {n} toxic and {n} non-toxic samples...")

balanced_df = pd.concat([
    toxic.sample(n, random_state=42),
    non_toxic.sample(n, random_state=42)
])

# 6. Shuffle and Finalize
# We shuffle the rows so the model doesn't see all toxic samples first
jigsaw_df = balanced_df[keep_cols].sample(frac=1, random_state=42).reset_index(drop=True)

# 7. Save using the '../' path for your 'notebooks' folder
output_path = '../data/raw/jigsaw_train.csv'
jigsaw_df.to_csv(output_path, index=False)

print("\nSuccess! Final Columns:", jigsaw_df.columns.tolist())
print("Label Distribution:")
print(jigsaw_df['toxic'].value_counts())

Creating a balanced dataset with 400 toxic and 400 non-toxic samples...

Success! Final Columns: ['comment_text', 'toxic']
Label Distribution:
toxic
0    400
1    400
Name: count, dtype: int64
